# 📋 SalesTeam AI — Feuille de Route & Tâches Restantes

> Ce notebook récapitule l'ensemble des tâches restantes à accomplir pour finaliser le système SalesTeam AI, avec l'explication détaillée, la priorité et l'objectif de chaque tâche.

---

## 📊 État d'Avancement Général du Projet

| Composant | Statut | Fichiers associés |
|---|---|---|
| **Couche 1 : Data Cleaning & Loader** | ✅ Terminé | `src/data/loader.py`, `src/data/cleaner.py` |
| **Couche 2 : Feature Engineering (24 features)** | ✅ Terminé | `src/features/feature_engineering.py` |
| **Couche 2 : Target Builder (Negative Sampling)** | ✅ Terminé | `src/models/target_builder.py` |
| **Couche 3 : Modèle XGBoost Classifier (ROC-AUC 0.99)** | ✅ Terminé | `src/models/train_classifier.py` |
| **Couche 3 : Modèle XGBoost Regressor (Quantités)** | ✅ Terminé | `src/models/train_regressor.py` |
| **Couche 4 : Service de Recommandation** | ✅ Terminé | `src/services/recommendation.py` |
| **Couche 5 : API REST FastAPI** | ✅ Terminé | `src/api/main.py`, `src/api/routes/` |
| **Interface Web de Test (React)** | ✅ Terminé | `frontend/src/App.jsx` |
| **Intégration Flutter (Mobile)** | 🔲 À faire | Application Mobile |
| **Boucle de Rétroaction (Feedback Loop)** | 🔲 À faire | `src/services/feedback.py` |
| **Génération Explications LLM (Mistral-7B)** | 🔲 À faire | `src/services/explanation.py` |
| **Filtrage Collaboratif (SVD)** | 🔲 À faire | `src/models/train_svd.py` |
| **Gestion du Cold Start** | 🔲 À faire | `src/models/cold_start.py` |
| **Conteneurisation Docker & Déploiement** | 🔲 À faire | `Dockerfile`, `docker-compose.yml` |

---

## 📱 Tâche 1 : Intégration de l'Application Mobile Flutter

**Priorité : 🔴 Haute** — C'est la tâche principale pour passer du prototype web au produit final terrain.

### 🎯 Objectif
Remplacer l'interface web de test (React) par l'intégration de l'API dans l'application mobile Flutter utilisée sur le terrain par les commerciaux.

### 📝 Explication Détaillée
Actuellement, l'interface web React (`frontend/src/App.jsx`) communique avec le backend FastAPI en envoyant des requêtes HTTP `POST` vers `http://127.0.0.1:8000/api/recommend`. Cette même logique doit être reproduite en **Dart** dans l'application Flutter :

1. **Créer un `ApiService` en Dart** — une classe qui encapsule tous les appels HTTP vers le backend.
2. **Envoyer une requête `POST /api/recommend`** contenant le JSON :
   ```json
   {
     "client_id": "CLT091206",
     "commercial_id": "COMMERCIAL_LSAT",
     "config": {
       "use_order_history": true,
       "use_seasonality": true,
       "nb_suggestions": 5
     }
   }
   ```
3. **Désérialiser la réponse JSON** en objets Dart (`ProductSuggestion`) pour alimenter la ListView du commercial.
4. **Envoyer le feedback** à `POST /api/feedback` quand le commercial valide sa commande.

### 🛠️ Fichiers à créer
- `lib/services/api_service.dart` — les appels HTTP.
- `lib/models/product_suggestion.dart` — le modèle Dart.
- `lib/screens/recommendation_screen.dart` — l'écran de recommandation.

### ⚠️ Point d'attention
- Sur un émulateur Android, l'URL du serveur local est `http://10.0.2.2:8000/api` (pas `127.0.0.1`).
- En production, utiliser un nom de domaine HTTPS sécurisé (ex: `https://api.salesteam.example.com/api`).

---

## 🔄 Tâche 2 : Boucle de Rétroaction (Feedback Loop) & Apprentissage Continu

**Priorité : 🟡 Moyenne** — Essentielle pour que l'IA s'améliore dans le temps, mais pas bloquante pour le MVP.

### 🎯 Objectif
Enregistrer les réactions du commercial (accepté, refusé, quantité modifiée) et ré-entraîner les modèles régulièrement pour que l'IA devienne de plus en plus intelligente.

### 📝 Explication Détaillée
À la fin de chaque visite client, le commercial valide sa commande finale. Ses choix (produits acceptés, refusés, quantités modifiées) sont envoyés à l'endpoint `POST /api/feedback`. Ce retour est stocké et utilisé pour :

- **Renforcer les bonnes prédictions** : Si le commercial accepte un produit recommandé, ce signal positif est ajouté au prochain cycle d'entraînement.
- **Pénaliser les mauvaises recommandations** : Si un produit est systématiquement refusé par le commercial, le modèle apprend à moins le suggérer.
- **Ajuster les quantités** : Si le commercial modifie régulièrement la quantité suggérée (ex: réduit de 10 à 3), le régresseur apprend de ces corrections.

### 🛠️ Actions à réaliser
1. Implémenter la sauvegarde dans `src/services/feedback.py` (fichiers CSV mensuels dans `data/feedback/`).
2. Créer un script `retrain.py` qui combine l'historique de ventes avec les retours des commerciaux.
3. Planifier un ré-entraînement automatique hebdomadaire (via un CRON job ou un script batch).
4. Mettre à jour les fichiers `.joblib` des modèles sans interrompre l'API (rechargement à chaud).

---

## 🤖 Tâche 3 : Génération des Explications en Langage Naturel (LLM / Mistral-7B)

**Priorité : 🟡 Moyenne** — Améliore considérablement l'expérience du commercial, mais le système fonctionne avec des explications basiques.

### 🎯 Objectif
Fournir des explications personnalisées et convaincantes en français pour chaque recommandation produit, générées par un modèle de langage (LLM).

### 📝 Explication Détaillée
Actuellement, les explications générées par `src/services/recommendation.py` utilisent des templates de texte basiques. L'idée est de passer à un LLM comme **Mistral-7B** (via HuggingFace Inference API) pour produire des justifications naturelles et contextuelles :

**Avant (template actuel) :**
> *"Probabilité d'achat élevée. Produit fréquemment commandé."*

**Après (avec LLM Mistral) :**
> *"Ce client commande habituellement 8 unités de ce modèle Samsung tous les 30 jours. Sa dernière commande remonte à 45 jours — il est très probablement en rupture de stock. Nous recommandons de lui proposer 10 unités pour couvrir la demande accumulée."*

### 🛠️ Actions à réaliser
1. Configurer la clé API HuggingFace dans `.env` (`HUGGINGFACE_API_KEY`).
2. Implémenter `src/services/explanation.py` avec un prompt engineeré en français.
3. Mettre en place un cache LRU en mémoire pour ne pas re-générer la même explication deux fois.
4. Prévoir un fallback vers les templates statiques si l'API HuggingFace est indisponible.

---

## 🧩 Tâche 4 : Modèle SVD & Filtrage Collaboratif (Collaborative Filtering)

**Priorité : 🟢 Basse** — Amélioration du système de recommandation, pas nécessaire pour le MVP.

### 🎯 Objectif
Ajouter une dimension d'apprentissage basée sur la **similitude entre clients** : *"Les clients qui ressemblent à ce client ont aussi acheté ce produit"*.

### 📝 Explication Détaillée
Actuellement, le classifieur XGBoost se base **uniquement** sur le comportement individuel du client (ses propres achats passés, sa fréquence, etc.). Il ne sait pas exploiter les patterns inter-clients.

Le modèle **SVD** (Singular Value Decomposition) décompose la matrice `Client × Produit` pour découvrir des motifs cachés. C'est la technique utilisée par Netflix et Amazon pour faire des recommandations de type *"Les utilisateurs qui ont acheté X ont aussi acheté Y"*.

### Comment ça marche
1. On construit une matrice `M[client][produit] = quantité_achetée`.
2. SVD factorise cette matrice en composantes latentes ("goûts cachés").
3. Le score SVD prédit la quantité que ce client achèterait pour un produit qu'il n'a jamais commandé.
4. On combine le score SVD avec le score XGBoost selon une pondération hybride :

$$\text{Score Final} = 0.6 \times \text{Score XGBoost} + 0.4 \times \text{Score SVD}$$

### 🛠️ Actions à réaliser
1. Implémenter `src/models/train_svd.py` (via `scikit-surprise` ou `scipy.sparse.linalg.svds`).
2. Créer `src/models/predictor.py` pour fusionner les scores Classifier + SVD.
3. Évaluer l'impact réel sur un jeu de test (A/B testing).

---

## ❄️ Tâche 5 : Gestion du Cold Start (Nouveaux Clients & Nouveaux Produits)

**Priorité : 🟡 Moyenne** — Critique pour le terrain car les commerciaux visitent régulièrement de nouveaux clients.

### 🎯 Objectif
Générer des recommandations pertinentes même lorsqu'un client est tout nouveau (aucun historique) ou lorsqu'un produit vient d'être lancé.

### 📝 Explication Détaillée
Quand un client est **nouveau** (jamais vu dans le dataset d'entraînement), XGBoost n'a **aucune donnée historique** pour lui. Toutes ses features (avg_qty, frequency, recency_days, etc.) sont à zéro — le modèle ne peut rien prédire de fiable.

Le module `src/models/cold_start.py` résout ce problème avec deux stratégies de **fallback** :

#### Stratégie A — Nouveau Client :
1. Utiliser les **coordonnées GPS** du nouveau client (si disponibles).
2. Trouver les **5 clients les plus proches géographiquement** via un K-Nearest Neighbors (BallTree sur latitude/longitude).
3. Recommander les **best-sellers de ces voisins** comme point de départ.

#### Stratégie B — Nouveau Produit :
1. Identifier la **catégorie** du nouveau produit (ex: "GSM SAMSUNG").
2. Identifier les clients qui achètent régulièrement cette catégorie.
3. Leur recommander le nouveau produit en priorité.

### 🛠️ Actions à réaliser
1. Implémenter `src/models/cold_start.py` avec les deux stratégies.
2. Brancher les fallbacks dans `src/services/recommendation.py` :
   - Si `client_id` n'est pas dans `training_set.csv` → appeler `cold_start.recommend_for_new_client()`.
   - Si `code_article` est récent → appeler `cold_start.recommend_new_product()`.

---

## 🐳 Tâche 6 : Conteneurisation Docker & Déploiement Cloud

**Priorité : 🟡 Moyenne** — Nécessaire pour passer en production, mais peut être fait en dernier.

### 🎯 Objectif
Empaqueter l'ensemble de l'application (API + Modèles IA) dans un conteneur Docker autonome et déployer sur un serveur cloud accessible par l'application Flutter.

### 📝 Explication Détaillée
Actuellement, l'API tourne en local avec `python -m uvicorn src.api.main:app --port 8000`. En production, il faut :

1. **Docker** garantit que l'application s'exécute de manière identique sur n'importe quel serveur, sans problème de versions Python ou de dépendances manquantes.
2. **Reverse Proxy (Nginx)** gère le HTTPS, le rate limiting et la compression gzip.
3. **CI/CD** automatise le redéploiement à chaque mise à jour du code.

### 🛠️ Actions à réaliser
1. Rédiger un `Dockerfile` optimisé (Python 3.10 slim + copie des modèles `.joblib`).
2. Rédiger un `docker-compose.yml` pour orchestrer l'API + Nginx.
3. Configurer HTTPS (Let's Encrypt ou Cloudflare Tunnel).
4. Créer un script `deploy.sh` pour automatiser le build & push vers le serveur.

### Exemple de Dockerfile
```dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src/ src/
COPY models/ models/
COPY data/processed/ data/processed/
EXPOSE 8000
CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

---

## 📌 Résumé — Ordre de Priorité Recommandé

| # | Tâche | Priorité | Impact |
|---|---|---|---|
| 1 | 📱 **Intégration Flutter** | 🔴 Haute | Permet l'utilisation sur le terrain |
| 2 | ❄️ **Cold Start** (Nouveaux clients) | 🟡 Moyenne | Couvre les cas où le modèle n'a pas d'historique |
| 3 | 🔄 **Feedback Loop** (Apprentissage continu) | 🟡 Moyenne | L'IA s'améliore avec chaque visite |
| 4 | 🐳 **Docker & Déploiement** | 🟡 Moyenne | Passage en production |
| 5 | 🤖 **Explications LLM** (Mistral-7B) | 🟡 Moyenne | Expérience utilisateur premium |
| 6 | 🧩 **SVD / Filtrage Collaboratif** | 🟢 Basse | Amélioration des recommandations inter-clients |

---

> **Note** : Toutes les couches terminées (Data → Features → Models → API → Web) sont déjà fonctionnelles et testées. Le système peut générer des recommandations en temps réel dès maintenant. Les tâches ci-dessus sont des améliorations pour passer du **prototype** au **produit de production**.

In [1]:
# Vérification rapide des artéfacts existants
import os

artifacts = {
    'Classifieur XGBoost':    'models/classifier_lsat.joblib',
    'Régresseur XGBoost':     'models/regressor_lsat.joblib',
    'Encodeur catégories':    'models/encoder_categorie.joblib',
    'Dataset entraînement':   'data/processed/training_set.csv',
    'Feature matrix':         'data/processed/feature_matrix.csv',
    'Main table':             'data/processed/main_table.csv',
}

print('=== VÉRIFICATION DES ARTÉFACTS ===')
for name, path in artifacts.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌'
    print(f'{status} {name}: {path}')

=== VÉRIFICATION DES ARTÉFACTS ===
✅ Classifieur XGBoost: models/classifier_lsat.joblib
✅ Régresseur XGBoost: models/regressor_lsat.joblib
✅ Encodeur catégories: models/encoder_categorie.joblib
✅ Dataset entraînement: data/processed/training_set.csv
✅ Feature matrix: data/processed/feature_matrix.csv
✅ Main table: data/processed/main_table.csv
